# Text Summarization Workflow Overview

This notebook provides an end-to-end executive and technical summary of **Task 1: Text Summarization**, detailing the pipeline methodology, data sources, LLM architecture choices, and specifically deep-diving into **Step 1: Prototyping and Validation**.

--- 

## 1. Pipeline Stages Overview

The text summarization pipeline maps raw, ultra-long congressional hearing transcripts into structured, entity-attributed summaries through five distinct stages:

1. **Stage 1: Prototyping & Draft Summarization** (`T1.1 - Process Prototyping and Evaluation/`)
   - **`T1.1.1 - Stage 1 - Preparation and Model Setup.py`**: Prepares paths, verifies scratch storage, loads datasets, and sets up model parameters in bfloat16 (`bf16`).
   - **`T1.1.2 - Stage 2 - Pipeline Execution and Output Generation.py`**: Runs turn-aware chunking and 3-prompt Map-Reduce LLM summarization on test benchmark samples, exporting output Parquet tables.

2. **Stage 2: Validation & Model Benchmark Comparison** (`T1.2` / Model Evaluation)
   - Benchmark tests 5 representative hearings across the length distribution (10th to 90th percentile) against candidate LLMs.
   - Evaluates wall-clock throughput ($s / 1,000$ words), token length adherence, and density of named claim/speaker attributions.

3. **Stage 3: Production Execution** (`T1.3 - Production Execution - SST Benchmark Pipeline.ipynb`)
   - Executes parallel, sharded batch summarization across all 2,352 hearings in the Space, Science, and Technology (SST) sample using the selected model (`Qwen/Qwen3.5-4B`).

4. **Stage 4: Post-Processing & Consolidation** (`T1.4 - Post-Processing - Summary Assembly & Storage.ipynb`)
   - Merges the 10 production shard files, enforces strict data completeness assertions, and exports the master `hearing_summaries.parquet` table.

5. **Stage 5: Quality Assessment & Evaluation** (`T1.5 - Quality Assessment - C-SPAN Discriminant Evaluation.ipynb`)
   - Evaluates summary specificity against independent human-written C-SPAN editorial abstracts using sentence embedding cosine similarity discriminant tests.

--- 

## 2. Models, Map-Reduce Architecture, and Data Traits

### Subsection 2.1: Map-Reduce Architecture using an LLM
Because hearing transcripts reach a median of **36,505 words (~49,000 tokens)** and a 90th percentile of **66,486 words (up to 376,848 words)**, single-pass LLM ingestion is impossible or results in severe "lost-in-the-middle" attention degradation. Map-reduce is executed across three dedicated prompts:
- **Map Prompt**: Reads raw transcript chunks $\to$ asks *"What was said by each speaker in this segment?"*
- **Collapse Prompt**: Condenses intermediate segment summaries if the accumulated map output exceeds context limits.
- **Reduce Prompt**: Reads all segment summaries $\to$ synthesizes the single final hearing summary, preserving speaker positions.

### Subsection 2.2: Turn-Aware Chunking Logic
Dividing a document arbitrarily across static character boundaries breaks speaker arguments in half. `L - Lib/L3 - LLM Orchestration and Summarization/chunking.py` implements **turn-aware chunking**:
- Chunks are assembled greedily out of **whole speaking turns** (uninterrupted participant contributions).
- Only turns exceeding the entire token budget are split along paragraph boundaries.
- Written Q&A ("questions for the record") are excluded as they were not spoken at the hearing.

### Subsection 2.3: Data Sources & Document Characteristics
- **GovInfo CHRG Hearings Corpus**: 36,378 packages (1995–2025). Requires preserving participant rosters (BioGuide IDs) and attributed statements.
- **C-SPAN Reference Abstracts**: 18,697 programs (median 51 words). Independent human-written reference summaries for discriminant quality testing.

### Subsection 2.4: LLM Candidate Models & Selection Rationale

Four candidate models were evaluated in unquantized 16-bit bfloat16 precision (`bf16`):

| Model Name | Parameter Count | VRAM Weight (bf16) | Throughput (s/1k words) | Checkable Attributed Claims | Selection Decision |
|---|---|---|---|---|---|
| **`Qwen/Qwen3.5-4B`** | 4.66B | 9.3 GB | 4.66 s | **57 claims** | **SELECTED FOR PRODUCTION**: Highest density of specific named claims, statutes, and figures. |
| **`meta-llama/Llama-3.2-3B-Instruct`** | 3.21B | 6.4 GB | **1.93 s** | 11 claims | Rejected: High speed, but vague summaries ("senators expressed concerns" without naming speakers). |
| **`google/gemma-4-E4B-it`** | 8.00B | 16.0 GB | ~6.20 s | 38 claims | Rejected: Slower throughput and lower claim density than Qwen. |
| **`meta-llama/Llama-3.1-8B-Instruct`** | 8.03B | 16.1 GB | ~8.10 s | 17 claims | Rejected: Higher memory footprint and lower attribution fidelity. |

--- 

## 3. Section 1: Prototyping and Validation Implementation

Stage 1 replaces monolithic notebooks with two dedicated Python scripts inside `T1.1 - Process Prototyping and Evaluation/`:

### Step 1: Preparation and Model Setup
- **Script**: [`T1.1.1 - Stage 1 - Preparation and Model Setup.py`](../T1.1%20-%20Process%20Prototyping%20and%20Evaluation/T1.1.1%20-%20Stage%201%20-%20Preparation%20and%20Model%20Setup.py)
- **What Was Done & Why**: Verifies scratch paths, initializes environment variables (`HF_HOME` pointing to `/scratch2/$USER/projects/llm_method/hf_cache`), loads the input transcript dataset (`hearings_sst.parquet`), and configures `Qwen/Qwen3.5-4B` in `bf16` precision.

### Step 2: Running Pipeline on Test Sample & Generating Output
- **Script**: [`T1.1.2 - Stage 2 - Pipeline Execution and Output Generation.py`](../T1.1%20-%20Process%20Prototyping%20and%20Evaluation/T1.1.2%20-%20Stage%202%20-%20Pipeline%20Execution%20and%20Output%20Generation.py)
- **What Was Done & Why**: Runs turn-aware chunking and executes the 3-stage Map-Reduce pipeline (Map $\to$ Collapse $\to$ Reduce) on test benchmark hearing transcripts. Formats and exports final summaries into Parquet tables (`output_test_summaries/test_hearing_summaries.parquet`).

--- 

## 4. Helper Functions Overview

All core algorithms and prompt orchestration code live inside `T1.X - Background Helper Functions/` (and `L - Lib/`):
- **`chunking.py`**: Turn-aware token chunking logic (`pack_turns_into_chunks`). Assembles whole speaking turns until the token budget is reached, preventing argument fragmentation.
- **`summarize.py`**: Prompt templates (`MAP_PROMPT`, `COLLAPSE_PROMPT`, `REDUCE_PROMPT`), LLM inference call loggers (`CallLog`), and map-reduce reducer functions.
- **`paths.py`**: Single source of path truth resolving scratch directories, Hugging Face caches, and repository outputs.